# Imports e Bibliotecas

In [137]:
import pandas as pd
import numpy as np
from enum import Enum
from math import floor as floor

# Dataset

In [138]:
deep_seek_v3_coleta = pd.read_csv("/content/Corpus_DeepSeek-V3.csv")
gpt_2_coleta = pd.read_csv("/content/Corpus_gpt-2.csv")
grok_1_coleta = pd.read_csv("/content/Corpus_grok-1.csv")
llama_coleta = pd.read_csv("/content/Corpus_llama.csv")
llama3_coleta = pd.read_csv("/content/Corpus_llama3.csv")

In [139]:
deep_seek_v3_coleta['Repositorio'] = 'DeepSeek_V3'
gpt_2_coleta['Repositorio'] = 'GPT_2'
grok_1_coleta['Repositorio'] = 'Grok_1'
llama_coleta['Repositorio'] = 'Llama'
llama3_coleta['Repositorio'] = 'Llama3'

In [140]:
todos = pd.concat([deep_seek_v3_coleta, gpt_2_coleta, grok_1_coleta, llama_coleta, llama3_coleta], ignore_index=True)

## Adicionando coluna $Validadas$
Representa se a *issue* está nos conjuntos das *issues* validadas como relacionadas à ética ou não

In [141]:
id_issues_validadas = pd.read_csv("/content/validadas.csv")

In [142]:
def addColunaValidada(dataset, id_issues_validadas):
  dataset['Validada'] = dataset['IdIssue'].isin(id_issues_validadas['IdIssue'])
  return dataset

In [143]:
todos = addColunaValidada(todos,id_issues_validadas)

## Identificando quem contribuiu para discussão

In [144]:
contribuintes = pd.read_csv('/content/ValidacaoExtra.csv')

In [145]:
contribuintes['Soma_Total'] = contribuintes['QtdComentarios'] + contribuintes['Info']
soma_agrupada = contribuintes.groupby(['IdIssue', 'Autor'])['Soma_Total'].sum().reset_index()
todos = todos.merge(
    soma_agrupada,
    left_on=['IdIssue', 'AutorComentario'],
    right_on=['IdIssue', 'Autor'],
    how='left'
)
todos.rename(columns={'Soma_Total': 'Volume'}, inplace=True)
todos['Volume'] = todos['Volume'].fillna(0)
todos.drop(columns=['Autor'], inplace=True)

## Anonimização

In [146]:
def anonimizaAutores(dataset):
  cp = dataset.copy()
  cp['AutorComentario'] = pd.factorize(cp['AutorComentario'])[0]
  cp['IdIssue'] = pd.factorize(cp['IdIssue'])[0]
  cp = cp.drop(columns=['LinkIssue','NumeroComentario','Comentario','Info'])
  return cp

### Split Dataset
1. Dataset somente com *issues* validadas (observa quem contribuiu ou não)
3. Dataset somente com *issues* invalidadas
---
> Permite avaliar se existe transversalidade fora do contexto de investigação de aspectos éticos (é mais um plus)

> Quero avaliar também a visão geral, ou seja, se um usuário que abordou aspecto ético também abordou outros temas não relacionados a aspectos éticos ao longo dos repositórios

> Vou analisar também o fato de ter participado numa discussão validada, se há a transversalidade

In [147]:
todos_anon=anonimizaAutores(todos)
todos_validadas=todos_anon[todos_anon['Validada']]
todos_invalidadas=todos_anon[todos_anon['Validada']==False]

In [148]:
todos.head(10)

,IdIssue,LinkIssue,Info,NumeroComentario,Comentario,AutorComentario,Termos_Info,Termos_Comentario,Termos,Repositorio,Validada,Volume
0,2759777407,https://github.com/deepseek-ai/DeepSeek-V3/iss...,are there any plans to release the knowledge d...,2601662103,,mowentian,,,,DeepSeek_V3,False,0.0
1,2759872971,https://github.com/deepseek-ai/DeepSeek-V3/pull/2,handle missing scale_inv_name>?<fixed an issue...,2562945271,i've uploaded the converted bf16 model here fo...,OpenSourceRonin,,,,DeepSeek_V3,False,0.0
2,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2580468057,other providers may have another notion of te...,Andrey36652,,,,DeepSeek_V3,True,1.0
3,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2564620466,"> hey, just a heads-up, this model isn’t meant...",BboyMadara,,,,DeepSeek_V3,True,0.0
4,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2564078228,could you attach full request in json format ...,Andrey36652,,"ethics, safety","ethics, safety",DeepSeek_V3,True,1.0
5,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2569139764,> > incorrect information. deepseek does suppo...,jasonfish568,,,,DeepSeek_V3,True,0.0
6,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2564028391,"hey, just a heads-up, this model isn’t meant f...",Value99,,,,DeepSeek_V3,True,0.0
7,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2578306124,"do you use it with deepseek api, or some othe...",Andrey36652,,,,DeepSeek_V3,True,1.0
8,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2577186033,> temperature 1.6-2.0. if you use really high ...,Happ1ness-dev,,,,DeepSeek_V3,True,0.0
9,2759906696,https://github.com/deepseek-ai/DeepSeek-V3/iss...,"everything is ok>?<as value99 said, since the ...",2579340891,"> other provider (together, fireworks, etc)\r\...",Happ1ness-dev,,,,DeepSeek_V3,True,0.0


In [149]:
todos_anon.head(10)

,IdIssue,AutorComentario,Termos_Info,Termos_Comentario,Termos,Repositorio,Validada,Volume
0,0,0,,,,DeepSeek_V3,False,0.0
1,1,1,,,,DeepSeek_V3,False,0.0
2,2,2,,,,DeepSeek_V3,True,1.0
3,2,3,,,,DeepSeek_V3,True,0.0
4,2,2,,"ethics, safety","ethics, safety",DeepSeek_V3,True,1.0
5,2,4,,,,DeepSeek_V3,True,0.0
6,2,5,,,,DeepSeek_V3,True,0.0
7,2,2,,,,DeepSeek_V3,True,1.0
8,2,6,,,,DeepSeek_V3,True,0.0
9,2,6,,,,DeepSeek_V3,True,0.0


# Estrutura de Dados

In [150]:
ids_validados = set(todos_validadas['IdIssue'].tolist())

In [151]:
ids_invalidados = set(todos_invalidadas['IdIssue'].tolist())

## Enum Repositórios

In [152]:
class Repositorios(Enum):
  DeepSeek_V3 = 1
  Grok_1 = 2
  GPT_2 = 3
  Llama = 4
  Llama3 = 5

In [153]:
#vou usar as validadas, a visualização pode colorir cada usuario com uma cor e o tamanho do no a qtd de comentarios feitos
#interessante montar rede por aspecto abordado ... ?
#o objetivo é ver a transversalidade do discurso

## Issue

In [154]:
class Issue:
  def __init__(self, id_issue:int, repositorio:Repositorios):
    self.id_issue = id_issue
    self.repositorio = repositorio
    self.relacionada_etica = self.verificaValidada()

  def verificaValidada(self):
    if self.id_issue in ids_validados:
      return True
    else:
      return False

## Heap

In [155]:
class Max_Heap:
  def __init__(self):
    self.issues = [None]
    self.tam = 0

  def pai(self, indice):
    return floor(indice/2)

  def esquerda(self, indice):
    return 2*indice

  def direita(self, indice):
    return 2*indice + 1

  def troca(self, indiceA, indiceB):
    aux = self.issues[indiceA]
    self.issues[indiceA] = self.issues[indiceB]
    self.issues[indiceB] = aux

  def max_heapify(self, indice):
    esq = self.esquerda(indice)
    dir = self.direita(indice)
    maior = indice

    if esq <= self.tam and self.issues[esq].id_issue > self.issues[maior].id_issue:
      maior = esq
    if dir <= self.tam and self.issues[dir].id_issue > self.issues[maior].id_issue:
      maior = dir

    if maior != indice:
      self.troca(indice, maior)
      self.max_heapify(maior)

  def constroi_max_heap(self, issues):
    tamanho_lista = len(issues)
    self.issues = [None]+list(issues)
    self.tam = tamanho_lista
    for i in range(floor(tamanho_lista/2),0,-1):
      self.max_heapify(i)

  def heap_max(self):
    return self.issues[1]

  def extraiMax(self):
    if self.tam < 1:
      return None

    max = self.issues[1]
    self.issues[1] = self.issues[self.tam]
    self.tam-=1
    self.issues.pop()
    if self.tam > 0:
      self.max_heapify(1)
    return max

  def insert(self, chave):
    self.tam += 1
    self.issues.append(chave)

    indice = self.tam
    p = self.pai(indice)

    while indice > 1 and self.issues[p].id_issue < self.issues[indice].id_issue:
        self.troca(indice, p)
        indice = p
        p = self.pai(indice)

## Nó Usuário

In [156]:
class No_Usuario:
  def __init__(self, id_participante:int, issues:list):
    self.id_participante = id_participante
    self.issues = issues
    self.repositorio_predominante = self.predominancia_repos() #indica o repo que um usuário mais participou ->
                                                                #pode ser bom ter uma lista de todos os repos q participou
    self.volume_discussao_aspectos_eticos = self.discussao_aspectos_eticos()

  def predominancia_repos(self):
    contabiliza_repo = {Repositorios(1).name:0,
                        Repositorios(2).name:0,
                        Repositorios(3).name:0,
                        Repositorios(4).name:0,
                        Repositorios(5).name:0}

    for i in self.issues:
      if i.repositorio.name in contabiliza_repo.keys():
        contabiliza_repo[i.repositorio.name]+=1

    maior = 0
    for i in contabiliza_repo.values():
      if maior < i:
        maior = i

    for chave, valor in contabiliza_repo.items():
      if valor == maior:
        chave_maior = chave

    return chave_maior

  def discussao_aspectos_eticos(self):
    #1.verificar se usuario esta em issue validada -> self.issues.id_issue in ids_validadas
    #2.nas issues validadas do usuario, buscar em todas_validadas o id_participante dele em AutorComentario
    #3.agrupar usuarios filtrados por AutorComentario e somar dado na coluna Volume em todas_validadas pelos agrupamentos
    #4.retornar resultado
    #parece IA, mas fui eu qm escreveu!!!

    ids_issues_usuario = [i.id_issue for i in self.issues]

    df_filtrado = todos_validadas[todos_validadas['IdIssue'].isin(ids_issues_usuario)]

    filtro_autor = df_filtrado['AutorComentario'] == self.id_participante
    df_filtrado = df_filtrado[filtro_autor]

    volume_total = int(df_filtrado['Volume'].sum())

    return volume_total


## Colocando dados num formato usável

In [157]:
issues = []
participantes = []

for row in todos_anon.itertuples(index=False):
  try:
      r = Repositorios[row.Repositorio]
  except KeyError:
      r = None
  issue = Issue(row.IdIssue,r)
  issues.append(issue)
  participante = No_Usuario(row.AutorComentario, [issue]) #juntar info de diferentes linhas de autorComentario em um unico nodo
  participantes.append(participante)

Como um mesmo usuário pode estar espalhado em várias issues, ele pode representar mais de um nó, o que não é interessante. Por isso, o método abaixo que encontra esses usuários espalhados e mescla suas informaçõe

In [158]:
def localizaMescla_participante_repetido(participantes):
    participantes_unicos = {}

    for participante in participantes:
        id_part = participante.id_participante

        if id_part not in participantes_unicos:
            # Antes de salvar o participante novo, removemos duplicatas internas de suas issues
            issues_limpas = []
            ids_vistos = set()

            for issue in participante.issues:
                if issue.id_issue not in ids_vistos:
                    issues_limpas.append(issue)
                    ids_vistos.add(issue.id_issue)

            # Atualiza a lista de issues do participante apenas com os itens únicos
            participante.issues = issues_limpas

            # Agora sim, salva no dicionário
            participantes_unicos[id_part] = participante

        else:
            # Lógica de mesclagem (agora funcionando sobre uma lista já limpa)
            p_existente = participantes_unicos[id_part]
            p_existente.volume_discussao_aspectos_eticos += participante.volume_discussao_aspectos_eticos

            ids_issues_existentes = {issue.id_issue for issue in p_existente.issues}

            for nova_issue in participante.issues:
                if nova_issue.id_issue not in ids_issues_existentes:
                    p_existente.issues.append(nova_issue)
                    ids_issues_existentes.add(nova_issue.id_issue)

    return list(participantes_unicos.values())

In [159]:
participantes_tratados = localizaMescla_participante_repetido(participantes)
for i in participantes_tratados:
  print('=====================================================================')
  print(f'Id_Participante={i.id_participante}')
  print(f'Volume={i.volume_discussao_aspectos_eticos}')
  print(f'Issues:')
  print('*********************************************************************')
  for j in i.issues:
    print(f'\tId Issue = {j.id_issue}')
    print(f'\tRepositório = {j.repositorio.name}')
    print(f'\tValidada = {j.relacionada_etica}')

  print('*********************************************************************')
  print('=====================================================================')

Id_Participante=0
Volume=2
Issues:
*********************************************************************
	Id Issue = 0
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 4
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 13
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 14
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 20
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 21
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 22
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 23
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 28
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 31
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 32
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 35
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 39
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 41
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 45
	Repositório = DeepSeek_V3
	Validada = Fal

Durante a construção das arestas, percebi o quão caro seria ligar os nós usando somente listas. Um nó se liga a outro se 2 usuários estão participando de uma mesma issue, ou seja, a issue é a aresta.

Se eu colocasse uma lista de usuários em Issue só faria uma dependência circular que queria evitar

Voltando, para popular a matriz de adjacência, seria preciso olhar o id da issue de um usuário e buscar na lista de issues de todos os outros usuários esse mesmo id para todas as issues. Mesmo que eu marcasse as issues já pesquisadas, isso ainda seria caro. Considerando todas as issues com um tamanho $x$ e $n$ a quantidade de usuários (nós), buscar por um id de issue linearmente seria demandaria uma quantidade absurda de operações.

Por isso, pensei em usar uma ED auxiliar para reduzir o custo dessa busca, tabela hash poderia ser complicado desenvolver uma função hash decente num tempo ábil. Pensei em árvore auto-balanceável pelo custo da busca no pior caso ser $O(log(n))$ e não seria preciso ordenar a entrada. Mas, um heap além de ser mais simples de implementar neste cenário, não perde em complexidade para a árvore, além de ser usado em algoritmos que atuam sobre grafos como Dijkstra

## Grafo

In [160]:
#matriz de adjacência, nós se conectam pela quantidade de issues em ambos. Se 2 usuarios participaram de 3 issues ao mesmo tempo, eles tem 3 conexoes
# a aresta pode ser rotulada com id do repo, visualmente pode ser uma cor
class Grafo:
  def __init__(self, lista_nos):
    self.lista_nos = lista_nos
    self.matrizAdjacencia = []
    self.inicializaMatrizAdjacencia()
    self.criar_aresta()

  def inicializaMatrizAdjacencia(self):
    n = len(self.lista_nos)
    self.matrizAdjacencia = [[[] for _ in range(n)] for _ in range(n)]

  def criar_aresta(self):
    dicionario_issues = {}
    for i in self.lista_nos:
      for j in i.issues:
        dicionario_issues[j.id_issue] = j

    h = Max_Heap()
    h.constroi_max_heap(list(dicionario_issues.values()))

    heaps_usuarios = [] #id usuario + respectivo heap

    for no in self.lista_nos:
      h_usuario = Max_Heap()
      h_usuario.constroi_max_heap(no.issues)
      elemento_usuario = {"id_participante": no.id_participante, "heap": h_usuario}
      heaps_usuarios.append(elemento_usuario)


    while h.tam > 0:
      m_val = h.extraiMax()
      m_id = m_val.id_issue
      nome_repo = m_val.repositorio.name

      usuariosM = []

      for i in range(len(heaps_usuarios)):
          while heaps_usuarios[i]['heap'].tam > 0:
              s = heaps_usuarios[i]['heap'].heap_max()

              if s.id_issue == m_id:
                  heaps_usuarios[i]['heap'].extraiMax()
                  usuariosM.append(i)
              else:
                break

      for i in range(len(usuariosM)):
          for j in range(i + 1, len(usuariosM)):
              u1 = usuariosM[i]
              u2 = usuariosM[j]

              aresta = (m_id, nome_repo)

              self.matrizAdjacencia[u1][u2].append(aresta)
              self.matrizAdjacencia[u2][u1].append(aresta)


  def vizinhos(self, id_participante):
    v = set()

    indice_alvo = -1
    for i, no in enumerate(self.lista_nos):
        if no.id_participante == id_participante:
            indice_alvo = i
            break

    if indice_alvo == -1:
        return v

    for j, arestas in enumerate(self.matrizAdjacencia[indice_alvo]):
      if len(arestas) > 0:
        id_vizinho = self.lista_nos[j].id_participante
        v.add(id_vizinho)
    return v


  def dfs(self, no, visitado):
    visitado[no.id_participante] = True
    #processar(no) #pode ser qualquer coisa, print, etc
    for w in self.vizinhos(no.id_participante):
      if not visitado[w]:
        self.dfs(w,visitado)

  def brandes(self):
    pass

In [161]:
g = Grafo(participantes_tratados)

# Análise